# Explore shot selections

Visually compare the per-pixel feature image produced by different `ShotSelection`
recipes, using `automask.viz`. A feature = a `reduction` (`mean`/`std`) over the
shots a `ShotSelection` keeps; see `automask/shot_selection.py` for the knobs
(`xray`, `laser`, `n_shots`, `filter_low`/`filter_high`, `normalization`).

**Kernel.** Select the **`Python (ana-psana)`** kernel (top-right). It's the
`ana-4.0.62` conda env — the only one with both `automask` *and* psana, and it
has the `SIT_*` data vars baked in, so both cached and uncached selections work.
Any kernel lacking `automask` fails at the import cell; one with `automask` but
no psana can only render selections already in the cache.

**Cache note.** A selection already in the FeatureStore cache renders instantly
(numpy only). A *new* selection triggers a raw-XTC pass (needs the psana kernel
above; first render is slow, then it's cached). The defaults below are all warm.

Present runs: **389** and **475**.

In [ ]:
import os
os.environ.setdefault("SIT_PSDM_DATA", "/Data/hippolyte.wallaert/psdm")
os.environ.setdefault("SIT_ROOT", "/Data/hippolyte.wallaert/psdm/sit_root")
os.environ.setdefault("SIT_DATA", "/Data/hippolyte.wallaert/psdm/data")

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

from automask import viz
from automask.shot_selection import ShotSelection

RUN = 389  # or 389

# Selections known to be warm in the cache (render numpy-only, no psana):
#   mean: xray=on, xray=on+ipm2, xray=off, xray=on trim=[0,0]
#   std : same set (xray=on, xray=on+ipm2, xray=off, xray=on trim=[0,0])

## 1. A single feature

`show_feature` accepts a catalogue name (`'umean'`, `'ustd'`, `'mean'`), a
`FeatureSpec`, or a bare `ShotSelection` (paired with `reduction`, default
`'mean'`). Dead/zero pixels are shown neutral; scale is a robust 1–99th pct.

In [ ]:
from automask.features import FeatureSpec

reductions = ["mean", "std", "median", "mad"]

selection = ShotSelection(
    xray="on",
    laser="on",
    n_shots=100,
    filter_low=0.03,
    filter_high=0.03,
    normalization="none"
)

feature_spec = FeatureSpec(
    name="on",
    selection=selection,
    reduction="mean"
)

In [ ]:
viz.show_feature(RUN, feature_spec)   # lit-beam per-pixel mean (xray=on)
plt.show()

In [ ]:
viz.show_feature(RUN, 'mean')   # lit-beam per-pixel mean (xray=on)
plt.show()

In [ ]:
viz.show_feature(RUN, 'ustd')   # lit-beam per-pixel mean (xray=on)
plt.show()

## 2. Effect of a shot selector: lit vs i0-normalized vs dark

`compare_selections` renders one panel per selection on a **shared** color scale
and colorbar, so brightness differences between recipes are real (not per-panel
autoscaled). Here: plain lit-beam mean vs the same with per-shot ipm2
normalization vs the beam-off dark.

In [ ]:
viz.compare_selections(
    RUN,
    [
        ShotSelection(xray='on'),                          # lit-beam mean
        ShotSelection(xray='on', normalization='ipm2'),    # + per-shot i0 scaling
        ShotSelection(xray='off'),                         # beam-off dark
    ],
    reduction='mean',
)
plt.show()

## 3. Same idea on the std reduction

The lit-beam per-pixel std (`ustd`) is where scattering contrast lives. Compare
plain vs ipm2-normalized.

In [ ]:
viz.compare_selections(
    RUN,
    [
        ShotSelection(xray='on'),
        ShotSelection(xray='on', normalization='ipm2'),
    ],
    reduction='std',
)
plt.show()

## Investigate from here

Swap in your own selections below. Any combination of the `ShotSelection` knobs
works; remember an **uncached** recipe needs the psana env (it does a raw-XTC
pass on first render, then caches). Ideas to try:

- intensity trim: `filter_low` / `filter_high` (e.g. `0.0` vs `0.1`) — how much
  do the brightest/dimmest shots move the mean?
- `laser='on'` vs `'off'` (pump state), or `n_shots=` to subsample.
- `reduction='std'` vs `'mean'`.

`show`, `show_feature`, and `compare_selections` all take an optional `ax=` and
return their artist/figure — pass `out='foo.png'` to save.

In [ ]:
viz.compare_selections(
    RUN,
    [
        # ShotSelection(xray="on", laser="on", n_shots=200), 
        # ShotSelection(xray="on", laser="on", n_shots=400), 
        ShotSelection(xray="on", laser="off", n_shots=800), 
        ShotSelection(xray="on", laser="off", n_shots=1600),
        #  ShotSelection(xray="on", laser="off", n_shots=3200)
    ],
    reduction='std',
)
plt.show()